# 이광수 페르소나 멀티에이전트 — 통합 파이프라인

RAG(지식) → StyleModel(말투) → Validator(검증) 을 **한 Colab 프로세스**에서 실행 + Gradio 데모.

**핵심 트릭:** 14B 하나로 LoRA 어댑터를 토글 — ON=말투 모델, OFF=순정 Qwen(내용 작성·검증). Gemini 불필요.

**준비(Google Drive `MyDrive/lgs/`에 둘 것):**
- `epochs2_lgs_style_lora_v2_qwen14b.zip` (StyleModel 어댑터)
- `secondary_sources.json` (24엔트리 KB)

**런타임:** A100 권장(14B + 임베딩 동시 로드).

In [ ]:
!pip -q install -U "transformers>=4.44" "peft>=0.12" "bitsandbytes>=0.43" accelerate sentence-transformers chromadb gradio
import torch; print('cuda', torch.cuda.is_available())

In [ ]:
# Drive 마운트 + 어댑터 압축해제 + 경로 확인
from google.colab import drive; drive.mount('/content/drive')
import zipfile, glob, os
LGS = '/content/drive/MyDrive/lgs'
KB_PATH = f'{LGS}/secondary_sources.json'        # 친일 KB
THOUGHT_PATH = f'{LGS}/thought_sources.json'     # 사상 KB (일반 모드용)
with zipfile.ZipFile(glob.glob(f'{LGS}/*qwen14b.zip')[0]) as z:
    z.extractall('/content/adapter')
ADAPTER = os.path.dirname(glob.glob('/content/adapter/**/adapter_config.json', recursive=True)[0])
print('친일KB:', os.path.exists(KB_PATH), '| 사상KB:', os.path.exists(THOUGHT_PATH))
print('adapter:', ADAPTER)

In [ ]:
# RAG: bge-m3 + ChromaDB 두 컬렉션 (친일 KB + 사상 KB)
import json, chromadb
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('BAAI/bge-m3')
client = chromadb.Client()
for c in ('lgs_chinil', 'lgs_thought'):   # 재실행 안전(있으면 삭제 후 재생성)
    try: client.delete_collection(c)
    except Exception: pass

# 친일 KB (친일 모드)
entries = json.load(open(KB_PATH, encoding='utf-8'))
def doc_text(e):
    return f"[{e['쟁점축']}] {e['하위주제']}\n{e['핵심_논거']}\n키워드: {', '.join(e.get('키워드',[]))}"
embs = embedder.encode([doc_text(e) for e in entries], normalize_embeddings=True)
coll = client.create_collection('lgs_chinil', metadata={'hnsw:space':'cosine'})
coll.add(ids=[e['id'] for e in entries], embeddings=[v.tolist() for v in embs],
         metadatas=[{'id':e['id'],'쟁점축':e['쟁점축']} for e in entries])
by_id = {e['id']: e for e in entries}

# 사상 KB (일반 모드)
th = json.load(open(THOUGHT_PATH, encoding='utf-8'))
def th_text(e):
    return f"[{e['주제']}] {e['하위주제']}\n{e['핵심_주장']}\n개념: {', '.join(e['핵심_개념'])}"
th_embs = embedder.encode([th_text(e) for e in th], normalize_embeddings=True)
coll_th = client.create_collection('lgs_thought', metadata={'hnsw:space':'cosine'})
coll_th.add(ids=[e['id'] for e in th], embeddings=[v.tolist() for v in th_embs],
            metadatas=[{'id':e['id'],'주제':e['주제']} for e in th])
by_id_th = {e['id']: e for e in th}
print('친일 KB:', coll.count(), '엔트리 | 사상 KB:', coll_th.count(), '엔트리')

In [ ]:
# 14B + StyleModel 어댑터 로드 (4비트)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
BASE = 'Qwen/Qwen2.5-14B-Instruct'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
        device_map='auto', torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, ADAPTER)
model.eval()
print('loaded 14B + adapter')

In [ ]:
# 생성 헬퍼 — 어댑터 ON/OFF 토글 + 중국어 누출 가드
import torch, re, json
from contextlib import nullcontext

def gen(msgs, use_adapter=True, max_new=320, temp=0.7, json_mode=False):
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors='pt', return_dict=True).to(model.device)
    ctx = nullcontext() if use_adapter else model.disable_adapter()
    with torch.no_grad(), ctx:
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=not json_mode,
                             temperature=temp, top_p=0.9)
    return tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# ── 중국어 누출 감지 + 자동 재생성 (Qwen 중국어 prior 대비) ──
_SIMP = set('来对们这为艺术唤启较够说话东车书长门问间吗呢吧给觉么样')  # 간체자(한국 한자엔 없음) 신호
def looks_chinese(t, min_hangul=0.40):
    if any(c in _SIMP for c in t):
        return True
    h = sum(0xAC00 <= ord(c) <= 0xD7A3 for c in t)      # 한글
    cjk = sum(0x4E00 <= ord(c) <= 0x9FFF for c in t)     # 한자/한문
    tot = h + cjk
    return bool(tot) and (h / tot) < min_hangul          # 한글 비율 너무 낮음 = 중국어 문장

def gen_ko(msgs, use_adapter=False, max_new=320, temps=(0.5, 0.3, 0.2), min_hangul=0.40):
    out = ''
    for t in temps:                                       # 누출이면 temp 낮춰 재생성
        out = gen(msgs, use_adapter=use_adapter, max_new=max_new, temp=t)
        if not looks_chinese(out, min_hangul):
            return out
    return out                                            # 끝까지 누출이면 마지막 결과라도 반환

# 표기 가드레일(프롬프트 차원) — compose/style 공통 삽입용
GUARD = ('[표기 규칙] 반드시 한국어로만 쓴다. 중국어 문장·간체자·중국어식 표현(而言, 对…而言 등)·로마자(munhak 등)를 '
         '절대 쓰지 마라. 한자는 한국 한자어를 정자(번체)로 괄호 병기할 때만 쓴다(예: 民族(민족), 內鮮一體(내선일체)). '
         '괄호 안에는 한글 독음만 넣는다. 조선인은 동포(同胞)·민족(民族)이라 부르고 통치자 어휘 "백성"은 쓰지 마라.')

STYLE_SYS = ('너는 1930~40년대 소설가 이광수(춘원)다. 입력 문장을 이광수 특유의 근대 국어(국한문혼용) 문체로 다시 써라. '
             '핵심 한자어는 한자(정자/번체)로 표기하고 괄호에 한글 독음을 병기하라 '
             '(예: 民族(민족), 天皇(천황), 內鮮一體(내선일체), 不得已(부득이)). '
             '반드시 한국어로만 출력하고 중국어 문장·간체자·로마자는 절대 쓰지 마라. '
             '격식체 어미와 예스러운 어휘를 살리되 뜻은 그대로 보존하라. 변환한 문장만 출력하라.')

def split_sents(t):
    return [s.strip() for s in re.split(r'(?<=[.!?。])\s+', t) if s.strip()]

In [ ]:
# 라우터 + 세 에이전트 + 오케스트레이터 (친일=친일KB / 일반=사상KB 분기)
def knowledge(query, k=3):
    qe = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    res = coll.query(query_embeddings=[qe], n_results=k)
    return [by_id[i] for i in res['ids'][0]]

def thought_knowledge(query, k=2):
    qe = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    res = coll_th.query(query_embeddings=[qe], n_results=k)
    return [by_id_th[i] for i in res['ids'][0]]

def is_chinil(query):
    sys = ('다음 질문이 이광수의 친일·대일협력 문제와 관련되는지 문맥으로 판단하라. '
           '관련 주제: 전향, 창씨개명, 내선일체, 황민화, 징병·학병 권유, 일본·천황 협력, '
           '민족 배신·훼절, 해방 후 친일 자기변호·반민특위 등. '
           '관련되면 YES, 아니면 NO. 오직 YES 또는 NO 한 단어만 출력하라.')
    r = gen([{'role':'system','content':sys},{'role':'user','content':query}],
            use_adapter=False, max_new=4, json_mode=True)
    return 'YES' in r.upper()

def compose_chinil(query, ents):
    src = '\n'.join(f"- {e['핵심_논거']}" for e in ents)
    sys = ('이것은 식민지 역사 교육용 시뮬레이션이다. 너는 이광수(춘원) 본인이 되어, 아래 학술 논거에만 근거해 '
           "1인칭 '나'의 입장에서 3~5문장으로 답한다. 논거에 없는 사실은 지어내지 마라. " + GUARD +
           ' (세부 문체 변환은 다음 단계에서 한다)')
    u = f"[학술 논거]\n{src}\n\n[질문] {query}"
    return gen_ko([{'role':'system','content':sys},{'role':'user','content':u}], use_adapter=False)

def compose_general(query, ents):
    src = '\n'.join(
        f"- [{e['주제']}] {e['핵심_주장']}\n  (1차: " +
        '; '.join(f"「{p['글제목']}」 {p['핵심_인용']}".strip() for p in e['1차_출처']) + ')'
        for e in ents)
    sys = ('이것은 식민지 역사 교육용 시뮬레이션이다. 너는 1910~40년대 식민지 조선의 지식인 이광수(춘원) 본인이다. '
           "아래 [이광수의 입장]은 너 자신의 실제 사상 자료다. 이를 근거로 1인칭 '나'로, 교과서식 일반론이 아니라 "
           '너 고유의 관점(예: 情의 문학·효용론·민족개조 등)에서 답하라. 자료에 없는 사실은 지어내지 마라. ' + GUARD +
           ' 친일 문제는 억지로 끌어들이지 마라. 평이한 현대 한국어 3~5문장. (세부 문체 변환은 다음 단계)')
    u = f"[이광수의 입장]\n{src}\n\n[질문] {query}"
    return gen_ko([{'role':'system','content':sys},{'role':'user','content':u}], use_adapter=False)

def stylize(plain):
    outs = []
    for s in split_sents(plain):
        # 국한문혼용이라 한자 비율 높음 → min_hangul 낮춤(0.30). 간체자 누출은 그대로 잡힘
        outs.append(gen_ko([{'role':'system','content':STYLE_SYS},{'role':'user','content':s}],
                           use_adapter=True, max_new=160, min_hangul=0.30))
    return ' '.join(outs)

def validate(answer, ents):
    # Lowell 도덕적 부조화 모델: 3단계 CoT (트리거30 / 기제40 / 설득력30 = 100, 70점 합격)
    src = '\n'.join(f"- {e['핵심_논거']}" for e in ents)
    sys = ('너는 인지부조화 이론(Cognitive Dissonance Theory)에 정통한 심리학자이자 역사학자다. '
           '이광수가 친일 행위로 인한 도덕적 부조화(Moral Dissonance)를 해소하려 어떤 자기합리화 전략을 쓰는지 '
           '아래 3단계로 평가하라.\n'
           '- Step1 트리거(0~30): 도덕적 찔림을 포착하고 "어쩔 수 없었다/시대의 흐름"식 외부 정당화로 책임을 회피하는가.\n'
           '- Step2 기제(0~40): ①합리화(친일을 민족개조·실력양성 명분으로 포장) ②피해자비난(조선 멸망을 조선인 탓) '
           '③자기확증(그럼에도 자신은 민족주의자라 주장)의 구사 정도.\n'
           '- Step3 설득력(0~30): 궤변이 그 자신에게 얼마나 완벽한 논리인가. 뻔뻔하고 치밀할수록 높게.\n'
           '총점 = 세 단계 합(0~100), 70점 이상 합격.\n'
           '반드시 아래 JSON만 출력: {"트리거":n,"기제":n,"설득력":n,"총점":n,'
           '"평가사유":"3단계 요약","피드백":"총점<70이면 어떤 기제를 강화할지 지시, 아니면 PASS"}')
    u = f"[논거]\n{src}\n\n[평가 대상 답변]\n{answer}"
    raw = gen([{'role':'system','content':sys},{'role':'user','content':u}],
              use_adapter=False, max_new=400, json_mode=True)
    try:
        return json.loads(re.search(r'\{.*\}', raw, re.S).group(0))
    except Exception:
        return {'총점': None, 'raw': raw}

def run_pipeline(query, k=3, retry=0):   # retry 인자는 호환용(현재 미사용)
    if is_chinil(query):
        ents = knowledge(query, k)
        plain = compose_chinil(query, ents)
        answer = stylize(plain)
        v = validate(answer, ents)
        sources = [{'id':e['id'],'쟁점축':e['쟁점축'],**e['논문_출처'],'하위주제':e['하위주제'],
                    '핵심_논거':e['핵심_논거']} for e in ents]
        return {'mode':'친일','answer':answer,'plain':plain,'sources':sources,'validation':v}
    else:
        tents = thought_knowledge(query, 2)
        plain = compose_general(query, tents)
        answer = stylize(plain)
        th_src = [{'주제':e['주제'],'하위주제':e['하위주제'],
                   '1차':' · '.join(p['글제목'] for p in e['1차_출처']),
                   '2차':'; '.join(f"{s['저자']}({s['연도']})" for s in e['2차_출처'])} for e in tents]
        return {'mode':'일반','answer':answer,'plain':plain,'sources':th_src,'validation':None}

In [ ]:
# 라우팅 테스트 — 일반/친일 모두 근거까지 출력
for q in ['문학이란 무엇입니까?', '창씨개명에 앞장선 까닭이 무엇입니까?']:
    r = run_pipeline(q)
    print('=' * 64)
    print('[질문]', q, '  →  모드:', r['mode'])
    print('[평이한 답변]', r['plain'])
    print('[이광수체]  ', r['answer'])
    if r['mode'] == '친일':
        print('[검증]', r['validation'])
        print('[학술 근거]')
        for s in r['sources']:
            print(f"   - [{s['쟁점축']}] {s['저자']}({s['연도']}) 「{s['제목'][:40]}」")
    else:
        print('[사상 근거]')
        for s in r['sources']:
            print(f"   - [{s['주제']}] {s['하위주제']}")
            print(f"        1차(원문): {s['1차']}")
            print(f"        2차(연구): {s['2차']}")
    print()

In [ ]:
# 8) 커스텀 HTML 프론트 — FastAPI(/ , /ask) + cloudflared (모드별 조건부 렌더링)
!pip -q install fastapi uvicorn nest-asyncio >/dev/null 2>&1
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

HTML = r'''<!DOCTYPE html><html lang="ko"><head><meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>춘원 이광수 페르소나</title><style>
*{box-sizing:border-box} body{margin:0;background:#faf7f1;color:#1c1a17;font-family:"Apple SD Gothic Neo","Malgun Gothic",system-ui,sans-serif;line-height:1.7}
.wrap{max-width:780px;margin:0 auto;padding:32px 22px 80px}
h1{font-size:24px;font-weight:800} .sub{color:#5d574e;font-size:14px;margin-bottom:24px}
.tag{display:inline-block;font-size:12px;color:#7c4a2d;background:#fff7e8;border:1px solid #efe0c6;padding:3px 10px;border-radius:999px;margin-bottom:10px}
.row{display:flex;gap:8px} input{flex:1;padding:12px 14px;border:1px solid #e7e1d6;border-radius:10px;font-size:15px}
button{padding:12px 20px;border:0;border-radius:10px;background:#7c4a2d;color:#fff;font-weight:700;cursor:pointer}
button:disabled{opacity:.5}
.card{background:#fffdf9;border:1px solid #e7e1d6;border-radius:14px;padding:20px;margin-top:20px}
.lgs{border-color:#c08a3e;background:#fff7e8;font-size:17px;white-space:pre-wrap}
.lbl{font-size:12px;letter-spacing:.1em;color:#b9874f;font-weight:700;margin-bottom:8px}
.src{border-bottom:1px solid #efe7d8;padding:10px 0} .src:last-child{border:0}
.chip{font-size:11px;background:#eef3f6;border:1px solid #dde7ec;color:#3f5560;padding:2px 8px;border-radius:999px}
.cite{font-weight:700;margin:4px 0 2px} .meta{font-size:12.5px;color:#5d574e}
details summary{cursor:pointer;font-size:12.5px;color:#7c4a2d;margin-top:4px}
details p{font-size:13px;color:#444;background:#f1ebe0;padding:10px;border-radius:8px;margin-top:6px}
.sc{display:flex;gap:14px;flex-wrap:wrap;margin:6px 0 8px} .sc span{font-size:13px}
.sc b{color:#7c4a2d} .loading{color:#b9874f}
.badge{display:inline-block;font-size:12px;padding:3px 10px;border-radius:999px;font-weight:700}
.badge.bad{background:#fbe6c4;border:1px solid #c08a3e;color:#7c4a2d}
.badge.gen{background:#eef3f6;border:1px solid #dde7ec;color:#3f5560}
.sub2{font-size:11px;color:#8a8278;margin:2px 0 4px}
</style></head><body><div class="wrap">
<span class="tag">식민지 근대 지식인 시뮬레이션 · 친일 맥락에서만 자기합리화 발동</span>
<h1>춘원 이광수 — 식민지 근대 지식인</h1>
<div class="sub">평소엔 자신의 사상(문학·민족·종교)에 근거해 답하고, 질문이 친일 맥락에 닿으면 자기합리화가 발동하며 학술 근거와 Lowell 도덕적 부조화 점수를 함께 보여줍니다.</div>
<div class="row"><input id="q" placeholder="예: 문학이란 무엇입니까? / 창씨개명에 앞장선 까닭은?" onkeydown="if(event.key==='Enter')ask()"><button id="b" onclick="ask()">묻기</button></div>
<div id="out"></div></div>
<script>
async function ask(){
 const q=document.getElementById('q').value.trim(); if(!q)return;
 const b=document.getElementById('b'), out=document.getElementById('out');
 b.disabled=true; out.innerHTML='<div class="card loading">이광수가 생각하는 중… (최대 1분)</div>';
 try{
  const r=await fetch('/ask',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({question:q})});
  if(!r.ok){ const t=await r.text(); out.innerHTML='<div class="card">서버 응답 '+r.status+'<br><small>'+t.slice(0,300).replace(/</g,'&lt;')+'</small></div>'; b.disabled=false; return; }
  const d=await r.json(); const v=d.validation||{}; const isC=(d.mode==='친일');
  const badge=isC?'<span class="badge bad">⚠ 자기합리화 감지 · 친일 맥락</span>':'<span class="badge gen">일반 대화 · 사상 근거</span>';
  let html=`<div class="card lgs">${badge}<div style="margin-top:10px">${d.answer||''}</div></div>`;
  if(isC){
   const pass=(v['총점']??0)>=70;
   html+=`<div class="card"><div class="lbl">자기합리화 평가 · Lowell 도덕적 부조화 모델</div>`+
     `<div class="sc"><span><b>총점 ${v['총점']??'-'}</b> / 100 ${pass?'✓ 합격':'✗ 미달(70)'}</span></div>`+
     `<div class="sub2">3단계 CoT (Step1 트리거 / Step2 기제: 합리화·피해자비난·자기확증 / Step3 설득력)</div>`+
     `<div class="sc"><span>트리거 <b>${v['트리거']??'-'}</b>/30</span><span>기제 <b>${v['기제']??'-'}</b>/40</span><span>설득력 <b>${v['설득력']??'-'}</b>/30</span></div>`+
     `<div class="meta">${v['평가사유']||''}</div></div>`;
   const src=(d.sources||[]).map(s=>`<div class="src"><span class="chip">${s['쟁점축']}</span><div class="cite">${s['저자']}(${s['연도']}) 「${s['제목']}」</div><div class="meta">${s['학술지']||''} ${s['권호']||''} · ${s['하위주제']||''}</div><details><summary>핵심 논거 보기</summary><p>${s['핵심_논거']||''}</p></details></div>`).join('');
   html+=`<div class="card"><div class="lbl">학술적 근거</div>${src}</div>`;
  } else {
   const tsrc=(d.sources||[]).map(s=>`<div class="src"><span class="chip">${s['주제']}</span><div class="cite">${s['하위주제']||''}</div><div class="meta">1차(원문): ${s['1차']||''}<br>2차(연구): ${s['2차']||''}</div></div>`).join('');
   if(tsrc) html+=`<div class="card"><div class="lbl">사상적 근거 (이광수 원문·연구)</div>${tsrc}</div>`;
  }
  out.innerHTML=html;
 }catch(e){ out.innerHTML='<div class="card">통신 오류(아마 타임아웃): '+e+'</div>'; }
 b.disabled=false;
}
</script></body></html>'''

from fastapi import FastAPI
from fastapi.responses import HTMLResponse, JSONResponse
from pydantic import BaseModel
import uvicorn, nest_asyncio, threading, subprocess, time, re, traceback

app = FastAPI()
@app.get('/')
def index(): return HTMLResponse(HTML)
class Q(BaseModel): question: str
@app.post('/ask')
def ask(q: Q):
    try:
        return JSONResponse(run_pipeline(q.question))
    except Exception as e:
        traceback.print_exc()
        return JSONResponse({'mode':'오류','answer':'(서버 오류) '+str(e), 'sources':[], 'validation':None})

nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning'), daemon=True).start()
time.sleep(4)
p = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:8000','--no-autoupdate'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
    if m: print('\n공개 URL (브라우저로 열기):', m.group(0)); break